In [1]:
import pandas as pd
import numpy as np
from faker import Faker 
import datetime as dt
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
def create_customer_data(n_rows: int, seed: int):
    """Generate synthetic customer records with reproducible random values.

    Args:
        n_rows: Number of customer records to generate.
        seed: Random seed for reproducibility.

    Returns:
        A DataFrame containing customer IDs, signup dates,
        acquisition sources,company size, industry, and countries.
    """
    fake = Faker(locale="en_US")
    fake.seed_instance(seed)
    rng = np.random.default_rng(seed)

    customer_ids = []
    signup_dates = []
    acquisition_channels = []
    industries = []
    company_sizes = []
    countries = []

    start_date = dt.date(2024, 8, 1)
    end_date = dt.date(2026, 8, 1)

    possible_countries = ["US", "Canada", "UK", "Australia", 
                          "Germany", "France"]

    possible_sources = ["Organic Search", "Paid Search", "Referral", 
                        "Social Media", "Partner", "Outbound Sales"]
    
    possible_industries = ["Technology", "Professional Services", 
                           "Retail", "Financial Services", 
                           "Healthcare", "Manufacturing", 
                           "Education", "Other"]

    possible_company_sizes = ["1-10", "11-50", "51-200", "201-500",
                              "501-1000", "1001+"]

    

    for _ in range(n_rows):
        customer_ids.append(fake.uuid4())

        signup_dates.append(fake.date_between(start_date=start_date, 
                                              end_date=end_date - dt.timedelta(days=1)))

        countries.append(rng.choice(a=possible_countries, 
                                          p=[0.44, 0.17, 0.12, 0.07,
                                             0.11, 0.09]))
        acquisition_channels.append(rng.choice(a=possible_sources, 
                                                     p=[0.22, 0.28, 
                                                        0.16, 0.12, 0.12, 
                                                        0.10]))

        industries.append(rng.choice(a=possible_industries, 
                                        p=[0.24, 0.15, 0.20, 0.12, 0.10, 
                                           0.09, 0.06, 0.04]))

        company_sizes.append(rng.choice(a=possible_company_sizes, 
                                              p=[0.22, 0.30, 0.24, 0.13, 
                                                 0.07, 0.04]))
                         
                                      

    cust_table = pd.DataFrame({
        "customer_id": customer_ids, 
        "signup_date": signup_dates, 
        "acquisition_channel": acquisition_channels, 
        "country": countries, 
        "industry": industries, 
        "company_size": company_sizes
    })

    return cust_table
        

In [3]:
customers = create_customer_data(1000, 42)

In [4]:
def create_subscription_data(customers, seed: int):
    """Generate one continuous subscription per customer.

    Args:
        customers: Customer DataFrame containing customer IDs
                   and signup dates.
        seed: Random seed for reproducibility.

    Returns:
        A DataFrame containing subscription records.
    """
    fake = Faker(locale="en_US")
    fake.seed_instance(seed)
    rng = np.random.default_rng(seed)

    subscription_ids = []
    customer_ids = []
    start_dates = []
    end_dates = []
    subscription_statuses = []
    billing_cycles = []
    plans = []
    monthly_prices = []

    possible_plans = ["Basic", "Pro", "Enterprise"]
    plan_prices = {"Basic": 29, "Pro": 79, "Enterprise": 199}
    possible_billing_cycles = ["Monthly", "Annual"]

    for _, customer in customers.iterrows():

        subscription_ids.append(fake.uuid4())
        customer_ids.append(customer["customer_id"])

        start_date = customer["signup_date"]
        start_dates.append(start_date)

        plan = rng.choice(a=possible_plans, p=[0.40, 0.45, 0.15])
        plans.append(plan)

        billing_cycle = rng.choice(a=possible_billing_cycles, 
                                         p=[0.70, 0.30])
        billing_cycles.append(billing_cycle)

        monthly_price = plan_prices[plan]

        if billing_cycle == "Annual": 
            monthly_price *= 0.85

        monthly_prices.append(round(monthly_price, 2))

        end_dates.append(None)
        subscription_statuses.append("Active")

    subscriptions_table = pd.DataFrame({
        "subscription_id": subscription_ids,
        "customer_id": customer_ids,
        "start_date": start_dates,
        "end_date": end_dates,
        "subscription_status": subscription_statuses,
        "billing_cycle": billing_cycles,
        "plan": plans,
        "monthly_price": monthly_prices
    })

    return subscriptions_table
    

In [5]:
subscriptions = create_subscription_data(customers, 42)

In [6]:
def create_subscription_events(subscriptions, seed: int):
    """Generate synthetic subscription plan-change events.

    Args:
        subscriptions: Subscription DataFrame.
        seed: Random seed for reproducibility.

    Returns:
        A DataFrame containing subscription plan-change events.
    """
    fake = Faker(locale="en_US")
    fake.seed_instance(seed)
    rng = np.random.default_rng(seed)

    event_ids = []
    subscription_ids = []
    event_dates = []
    event_types = []
    previous_plans = []
    new_plans = []
    previous_prices = []
    new_prices = []

    plan_prices = {
        "Basic": 29,
        "Pro": 79,
        "Enterprise": 199
    }

    transition_probabilities = {
        "Basic": {"Pro": 0.80, "Enterprise": 0.20},
        "Pro": {"Basic": 0.20, "Enterprise": 0.80},
        "Enterprise": {"Basic": 0.10, "Pro": 0.90}
    }

    observation_end = dt.date(2026, 8, 1)

    for _, subscription in subscriptions.iterrows():

        start_date = pd.to_datetime(subscription["start_date"]).date()

        end_date = observation_end

        earliest_event_date = start_date + dt.timedelta(days=30)
        latest_event_date = end_date - dt.timedelta(days=1)

        if earliest_event_date > latest_event_date:
            continue

        eligible_days = (latest_event_date - earliest_event_date).days + 1

        n_events = min(
            rng.poisson(lam=(eligible_days / 365) * 0.40),
            3
        )

        if n_events == 0:
            continue

        possible_dates = np.arange(eligible_days)

        selected_dates = np.sort(rng.choice(
            a=possible_dates,
            size=min(n_events, eligible_days),
            replace=False
            )
        )

        current_plan = subscription["plan"]

        billing_multiplier = (
            0.85 if subscription["billing_cycle"] == "Annual" else 1
        )

        for day_offset in selected_dates:

            event_date = earliest_event_date + dt.timedelta(
                days=int(day_offset)
            )

            available_plans = list(transition_probabilities[current_plan].keys())

            probabilities = list(transition_probabilities[current_plan].values())

            next_plan = rng.choice(a=available_plans, p=probabilities)

            previous_price = round(
                plan_prices[current_plan] * billing_multiplier, 2
            )

            next_price = round(
                plan_prices[next_plan] * billing_multiplier, 2
            )

            event_type = ("Upgrade" if next_price > previous_price else "Downgrade")

            event_ids.append(fake.uuid4())
            subscription_ids.append(subscription["subscription_id"])
            event_dates.append(event_date)
            event_types.append(event_type)
            previous_plans.append(current_plan)
            new_plans.append(next_plan)
            previous_prices.append(previous_price)
            new_prices.append(next_price)

            current_plan = next_plan
            
    subscription_events = pd.DataFrame({
        "event_id": event_ids,
        "subscription_id": subscription_ids,
        "event_date": event_dates,
        "event_type": event_types,
        "previous_plan": previous_plans,
        "new_plan": new_plans,
        "previous_price": previous_prices,
        "new_price": new_prices
    })

    return subscription_events  

In [7]:
subscription_events = create_subscription_events(subscriptions, 42)

In [8]:
subscription_events["subscription_id"].isin(subscriptions["subscription_id"]).all()

True

In [9]:
subscription_events["event_id"].is_unique

True

In [10]:
merged = subscription_events.merge(
    subscriptions[["subscription_id", "start_date"]],
    on="subscription_id",
    how="left",
    validate="many_to_one",
)

assert (merged["event_date"] > merged["start_date"]).all()

In [11]:
merged = subscription_events.merge(
    subscriptions[["subscription_id", "end_date"]].dropna(subset=["end_date"]),
    on="subscription_id",
    how="inner",
    validate="many_to_one",
)

assert (merged["event_date"] < merged["end_date"]).all()

In [12]:
def create_usage_events(customers, subscriptions, seed: int):
    """Generate synthetic SaaS product usage events.

    Args:
        customers: Customer DataFrame containing customer IDs
                   and company sizes.
        subscriptions: Subscription DataFrame containing customer IDs,
                       start dates, end dates, and plans.
        seed: Random seed for reproducibility.

    Returns:
        A DataFrame containing product usage events.
    """

    fake = Faker(locale="en_US")
    fake.seed_instance(seed)
    rng = np.random.default_rng(seed)

    event_ids = []
    customer_ids = []
    event_timestamps = []
    event_types = []
    feature_categories = []
    session_durations = []

    possible_events = [
        "Dashboard Viewed",
        "Report Generated",
        "Data Exported",
        "Integration Connected",
        "Integration Used",
        "Team Member Invited",
        "Custom Metric Created",
        "Alert Configured"
    ]

    event_probabilities = [
        0.38, 0.18, 0.08, 0.04,
        0.12, 0.05, 0.08, 0.07
    ]

    feature_mapping = {
        "Dashboard Viewed": "Dashboard",
        "Report Generated": "Reporting",
        "Data Exported": "Reporting",
        "Integration Connected": "Integrations",
        "Integration Used": "Integrations",
        "Team Member Invited": "Collaboration",
        "Custom Metric Created": "Analytics",
        "Alert Configured": "Analytics"
    }

    duration_ranges = {
        "Dashboard Viewed": (60, 900),
        "Report Generated": (120, 1200),
        "Data Exported": (30, 600),
        "Integration Connected": (60, 900),
        "Integration Used": (30, 600),
        "Team Member Invited": (30, 300),
        "Custom Metric Created": (60, 900),
        "Alert Configured": (30, 600)
    }

    plan_activity = {
        "Basic": 0.12,
        "Pro": 0.17,
        "Enterprise": 0.22
    }

    company_size_activity = {
        "1-10": 0.90,
        "11-50": 1.00,
        "51-200": 1.05,
        "201-500": 1.10,
        "501-1000": 1.15,
        "1001+": 1.20
    }

    observation_end = dt.date(2026, 8, 1)

    subscription_data = subscriptions.merge(right=customers[["customer_id", "company_size"]],
                                            how="left", on="customer_id", validate="one_to_one")

    for _, subscription in subscription_data.iterrows():
        customer_id = subscription["customer_id"]

        start_date = pd.to_datetime(subscription["start_date"]).date()

        if pd.isna(subscription["end_date"]):
            end_date = observation_end
        else:
            end_date = pd.to_datetime(subscription["end_date"]).date()


        last_active_date = min(end_date - dt.timedelta(days=1), 
                               observation_end - dt.timedelta(days=1))

        if start_date > last_active_date:
            continue


        engagement_multiplier = rng.uniform(low=0.70, high=1.30)

        active_day_probability = (
            plan_activity[subscription["plan"]]
            * company_size_activity[subscription["company_size"]]
            * engagement_multiplier
        )

        active_day_probability = np.clip(active_day_probability, 0.05, 0.50)

        periods = pd.period_range(start=start_date, end=last_active_date, freq="M")

        for period in periods:
            month_start = period.start_time.date()
            month_end = period.end_time.date()

            period_start = max(start_date, month_start)
            period_end = min(last_active_date, month_end)

            days_in_period = (period_end - period_start).days + 1

            n_active_days = rng.binomial(n=days_in_period, 
                                               p=active_day_probability)

            if n_active_days == 0:
                continue

            possible_days = pd.date_range(start=period_start, 
                                          end=period_end, 
                                          freq="D").date

            active_days = rng.choice(a=possible_days, 
                                           size=n_active_days, 
                                           replace=False)

            for active_day in sorted(active_days):
                n_events = int(rng.integers(1, 4))

                for _ in range(n_events):
                    event_type = rng.choice(a=possible_events, 
                                                  p=event_probabilities)

                    feature_category = feature_mapping[event_type]

                    duration_min, duration_max = (duration_ranges[event_type])

                    session_duration = int(rng.integers(duration_min, duration_max + 1))

                    hour = int(rng.integers(7, 21))
                    minute = int(rng.integers(0, 60))
                    second = int(rng.integers(0, 60))

                    event_timestamp = dt.datetime.combine(active_day, dt.time(hour=hour, minute=minute, second=second))

                    event_ids.append(fake.uuid4())
                    customer_ids.append(customer_id)
                    event_timestamps.append(event_timestamp)
                    event_types.append(event_type)
                    feature_categories.append(feature_category)
                    session_durations.append(session_duration)

        

    usage_events = pd.DataFrame({
        "event_id": event_ids,
        "customer_id": customer_ids,
        "event_timestamp": event_timestamps,
        "event_type": event_types,
        "feature_category": feature_categories,
        "session_duration": session_durations
    })

    usage_events = usage_events.sort_values(by=["customer_id", 
                                                "event_timestamp"]).reset_index(drop=True)
    return usage_events


In [13]:
usage = create_usage_events(
    customers=customers,
    subscriptions=subscriptions,
    seed=42
)

In [14]:
usage.head()


,event_id,customer_id,event_timestamp,event_type,feature_category,session_duration
0,05764a93-a61f-48fc-b027-e4cb15780574,004b6fab-fcf5-4188-932e-6dcd83bc9478,2025-12-07 09:50:18,Dashboard Viewed,Dashboard,842
1,6ca4c622-ee06-4a2d-9801-53b01496059f,004b6fab-fcf5-4188-932e-6dcd83bc9478,2025-12-07 16:35:29,Report Generated,Reporting,381
2,c87faf6b-2940-49b8-9b4f-50653a070c29,004b6fab-fcf5-4188-932e-6dcd83bc9478,2025-12-21 08:23:52,Integration Used,Integrations,440
3,a08a93ba-ca2c-4be2-bd31-b02d2007b338,004b6fab-fcf5-4188-932e-6dcd83bc9478,2025-12-21 14:35:25,Dashboard Viewed,Dashboard,421
4,6717bc17-f6ca-4c0b-9f68-1f0e7b268b93,004b6fab-fcf5-4188-932e-6dcd83bc9478,2025-12-21 18:37:13,Alert Configured,Analytics,291


In [15]:
usage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117583 entries, 0 to 117582
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   event_id          117583 non-null  object        
 1   customer_id       117583 non-null  object        
 2   event_timestamp   117583 non-null  datetime64[ns]
 3   event_type        117583 non-null  object        
 4   feature_category  117583 non-null  object        
 5   session_duration  117583 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 5.4+ MB


In [16]:
usage.describe()

,event_timestamp,session_duration
count,117583,117583.000000
mean,2025-11-28 17:53:27.892620288,450.670454
min,2024-08-02 09:23:30,30.000000
25%,2025-07-28 17:06:20.500000,228.000000
50%,2025-12-29 07:59:48,419.000000
75%,2026-04-23 18:52:29.500000,636.000000
max,2026-07-31 20:51:36,1200.000000
std,NaN,268.448904


In [17]:
usage["event_type"].value_counts(normalize=True)

event_type
Dashboard Viewed         0.380820
Report Generated         0.178869
Integration Used         0.120681
Data Exported            0.080845
Custom Metric Created    0.079255
Alert Configured         0.070622
Team Member Invited      0.049684
Integration Connected    0.039223
Name: proportion, dtype: float64

In [18]:
usage["feature_category"].value_counts()

feature_category
Dashboard        44778
Reporting        30538
Integrations     18802
Analytics        17623
Collaboration     5842
Name: count, dtype: int64

In [19]:
usage["event_id"].is_unique

True

In [20]:
usage.isna().sum()

event_id            0
customer_id         0
event_timestamp     0
event_type          0
feature_category    0
session_duration    0
dtype: int64

In [21]:
usage["customer_id"].isin(customers["customer_id"]).all()

True

In [22]:
usage[
    usage["event_timestamp"] >= pd.Timestamp("2026-08-01")
]

,event_id,customer_id,event_timestamp,event_type,feature_category,session_duration


In [23]:
def create_invoice_data(customers, subscriptions,
                        subscription_events,
                        seed: int):
    """Generate synthetic subscription invoice records.

    Args:
        subscriptions: Subscription DataFrame.
        subscription_events: Subscription lifecycle events DataFrame.
        seed: Random seed for reproducibility.

    Returns:
        A DataFrame containing invoice records.
    """
    fake = Faker(locale="en_US")
    fake.seed_instance(seed)
    rng = np.random.default_rng(seed)

    invoice_ids = []
    subscription_ids = []
    invoice_dates = []
    due_dates = []
    amounts = []
    payment_statuses = []
    payment_dates = []
    retry_counts = []
    failure_reasons = []
    recovered_values = []
    last_attempt_dates = []

    plan_prices = {
        "Basic": 29,
        "Pro": 79,
        "Enterprise": 199
    }

    observation_end = dt.date(2026, 8, 1)

    failure_rates = {
        "US": {"Monthly": 0.050, "Annual": 0.025},
        "Canada": {"Monthly": 0.040, "Annual": 0.020},
        "UK": {"Monthly": 0.080, "Annual": 0.030},
        "Australia": {"Monthly": 0.050, "Annual": 0.025},
        "Germany": {"Monthly": 0.040, "Annual": 0.020},
        "France": {"Monthly": 0.050, "Annual": 0.025}
    }

    possible_failure_reasons = [
        "Insufficient Funds",
        "Expired Card",
        "Bank Declined",
        "Payment Method Invalid"
    ]

    events_by_subscription = {
        subscription_id: group.sort_values("event_date")
        for subscription_id, group in subscription_events.groupby("subscription_id")}

    country_by_customer = customers.set_index(keys="customer_id")["country"].to_dict()


    for _, subscription in subscriptions.iterrows():

        subscription_id = subscription["subscription_id"]

        country = country_by_customer[subscription["customer_id"]]
        billing_cycle = subscription["billing_cycle"]

        start_date = pd.to_datetime(subscription["start_date"]).date()

        last_invoice_date = observation_end - dt.timedelta(days=1)

        billing_interval = (1 if billing_cycle == "Monthly" else 12)

        events = events_by_subscription.get(subscription_id, pd.DataFrame())

        if not events.empty:
            events = events.copy()
            events["event_date"] = pd.to_datetime(events["event_date"]).dt.date

        current_plan = subscription["plan"]
        event_index = 0
        invoice_number = 0

        while True:
            invoice_date = (pd.Timestamp(start_date) + pd.DateOffset(months=invoice_number * billing_interval)).date()

            if invoice_date > last_invoice_date:
                break

            while (
                event_index < len(events)
                and events.iloc[event_index]["event_date"] <= invoice_date
            ):
                current_plan = events.iloc[event_index]["new_plan"]
                event_index += 1

            amount = plan_prices[current_plan]

            if billing_cycle == "Annual":
                amount *= 12 * 0.85
        

            amount = round(amount, 2)

            due_date = invoice_date + dt.timedelta(days=14)

            payment_status = None
            payment_date = None
            retry_count = 0
            failure_reason = None
            recovered = False
            last_attempt_date = None

            if invoice_date + dt.timedelta(days=30) > observation_end:
                payment_status = "Pending"

            else:
                failure_probability = failure_rates[country][billing_cycle]

                initial_failure = (
                    rng.random() < failure_probability
                )

                if not initial_failure:
                    payment_status = "Paid"

                    payment_delay = int(rng.integers(0, 8))
                    payment_date = invoice_date + dt.timedelta(
                        days=payment_delay
                    )

                    last_attempt_date = payment_date

                else:
                    failure_reason = rng.choice(
                        possible_failure_reasons
                    )

                    current_attempt_date = invoice_date
                    payment_status = "Failed"

                    # Up to three additional attempts.
                    for attempt in range(1, 4):
                        retry_count = attempt

                        retry_delay = int(rng.integers(3, 8))
                        current_attempt_date += dt.timedelta(
                            days=retry_delay
                        )

                        last_attempt_date = current_attempt_date

                        # Each retry has a chance of recovering payment.
                        if rng.random() < 0.42:
                            payment_status = "Paid"
                            recovered = True
                            payment_date = current_attempt_date
                            break

                    # A final failed payment remains unpaid.
                    if payment_status == "Failed":
                        payment_date = None

            invoice_ids.append(fake.uuid4())
            subscription_ids.append(subscription_id)
            invoice_dates.append(invoice_date)
            due_dates.append(due_date)
            amounts.append(amount)
            payment_statuses.append(payment_status)
            payment_dates.append(payment_date)
            retry_counts.append(retry_count)
            failure_reasons.append(failure_reason)
            recovered_values.append(recovered)
            last_attempt_dates.append(last_attempt_date)
            

            invoice_number += 1

    invoices = pd.DataFrame({
        "invoice_id": invoice_ids,
        "subscription_id": subscription_ids,
        "invoice_date": invoice_dates,
        "due_date": due_dates,
        "amount": amounts,
        "payment_status": payment_statuses,
        "payment_date": payment_dates,
        "retry_count": retry_counts,
        "failure_reason": failure_reasons,
        "recovered": recovered_values,
        "last_attempt_date": last_attempt_dates
    })

    return invoices
    

In [24]:
invoices = create_invoice_data(customers=customers,
    subscriptions=subscriptions,
    subscription_events=subscription_events,
    seed=42
)

In [25]:
invoices.head()

,invoice_id,subscription_id,invoice_date,due_date,amount,payment_status,payment_date,retry_count,failure_reason,recovered,last_attempt_date
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-02-17,2025-03-03,79.0,Paid,2025-02-22,0,None,False,2025-02-22
1,23b8c1e9-3924-46de-beb1-3b9046685257,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-03-17,2025-03-31,79.0,Paid,2025-03-20,0,None,False,2025-03-20
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-04-17,2025-05-01,79.0,Paid,2025-04-18,0,None,False,2025-04-18
3,972a8469-1641-4f82-8b9d-2434e465e150,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-05-17,2025-05-31,79.0,Paid,2025-05-17,0,None,False,2025-05-17
4,17fc695a-07a0-4a6e-8822-e8f36c031199,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-06-17,2025-07-01,79.0,Paid,2025-06-22,0,None,False,2025-06-22


In [26]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9334 entries, 0 to 9333
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   invoice_id         9334 non-null   object 
 1   subscription_id    9334 non-null   object 
 2   invoice_date       9334 non-null   object 
 3   due_date           9334 non-null   object 
 4   amount             9334 non-null   float64
 5   payment_status     9334 non-null   object 
 6   payment_date       8569 non-null   object 
 7   retry_count        9334 non-null   int64  
 8   failure_reason     426 non-null    object 
 9   recovered          9334 non-null   bool   
 10  last_attempt_date  8661 non-null   object 
dtypes: bool(1), float64(1), int64(1), object(8)
memory usage: 738.5+ KB


In [27]:
invoices.describe()

,amount,retry_count
count,9334.000000,9334.000000
mean,122.454639,0.087958
std,220.264936,0.442418
min,29.000000,0.000000
25%,29.000000,0.000000
50%,79.000000,0.000000
75%,199.000000,0.000000
max,2029.800000,3.000000


In [28]:
invoices["payment_status"].value_counts(normalize=True)

payment_status
Paid       0.918042
Pending    0.072102
Failed     0.009856
Name: proportion, dtype: float64

In [29]:
invoices["invoice_id"].is_unique

True

In [30]:
invoices["subscription_id"].isin(subscriptions["subscription_id"]).all()

True

In [31]:
(invoices["due_date"] - invoices["invoice_date"] == pd.Timedelta(days=14)).all()

True

In [32]:
invoices[invoices["payment_status"] == "Failed"]["payment_date"].isna().all()

True

In [33]:
invoices[invoices["payment_status"] == "Overdue"]["payment_date"].isna().all()

True

In [34]:
(invoices["payment_date"] < invoices["invoice_date"]).all()

False

In [35]:
def create_support_tickets(customers,
                           subscriptions,
                           seed: int):
    """Generate synthetic customer support ticket records.

    Args:
        customers: Customer DataFrame.
        subscriptions: Subscription DataFrame.
        seed: Random seed for reproducibility.

    Returns:
        A DataFrame containing support ticket records.
    """
    fake = Faker(locale="en_US")
    fake.seed_instance(seed)
    rng = np.random.default_rng(seed)

    ticket_ids = []
    customer_ids = []
    created_timestamps = []
    resolved_timestamps = []
    issue_categories = []
    priorities = []
    ticket_statuses = []
    resolution_times = []
    sla_hours_values = []
    sla_breached_values = []

    observation_end = dt.date(2026, 8, 1)

    plan_ticket_rate = {
        "Basic": 0.8,
        "Pro": 1.2,
        "Enterprise": 1.8
    }

    company_size_multiplier = {
        "1-10": 0.90,
        "11-50": 1.00,
        "51-200": 1.05,
        "201-500": 1.10,
        "501-1000": 1.15,
        "1001+": 1.20
    }    

    categories = [
        "Technical Issue", "Billing", "Account Management",
        "Integration", "Product Question","Data Issue"
    ]

    category_probabilities = [0.25, 0.15, 0.12, 0.18, 0.18, 0.12]

    priority_options = ["Low", "Medium", "High", "Critical"]

    priority_probabilities = [0.25, 0.50, 0.20, 0.05]

    priority_sla = {"Low": 72, 
                    "Medium": 48, 
                    "High": 24, 
                    "Critical": 12
                   }
    
    category_sla_multiplier = {
        "Billing": 0.75,
        "Account Management": 0.85,
        "Product Question": 0.90,
        "Technical Issue": 1.20,
        "Integration": 1.35,
        "Data Issue": 1.50
    }

    unresolved_probability = {
        "Billing": 0.03,
        "Account Management": 0.03,
        "Product Question": 0.04,
        "Technical Issue": 0.10,
        "Integration": 0.13,
        "Data Issue": 0.15
    }

    subscription_data = subscriptions.merge(
        customers[["customer_id", "company_size"]],
        on="customer_id", how="left", validate="one_to_one"
    )

    for _, subscription in subscription_data.iterrows():
        customer_id = subscription["customer_id"]

        start_date = pd.to_datetime(subscription["start_date"]).date()

        last_ticket_date = observation_end - dt.timedelta(days=1)

        active_days = (last_ticket_date - start_date).days + 1

        if active_days <= 0:
            continue

        annual_ticket_rate = (
            plan_ticket_rate[subscription["plan"]]
            * company_size_multiplier[subscription["company_size"]]
        )

        support_propensity = rng.lognormal(mean=0, sigma=0.35)

        technical_issue_propensity = rng.lognormal(mean=0, sigma=0.65)

        expected_tickets = (
            (active_days / 365)
            * annual_ticket_rate
            * support_propensity
        )

        number_of_tickets = rng.poisson(expected_tickets)

        for _ in range(number_of_tickets):
            days_since_start = int(rng.integers(0, active_days))

            created_date = (start_date + dt.timedelta(days=days_since_start))

            created_time = dt.datetime.combine(
                created_date,
                dt.time(
                    hour=int(rng.integers(7, 21)),
                    minute=int(rng.integers(0, 60))
                )
            )

            customer_category_probabilities = np.array(category_probabilities.copy(), dtype=float)

            customer_category_probabilities[0] *= (
                technical_issue_propensity
            )
            
            customer_category_probabilities = (
                customer_category_probabilities
                / np.sum(customer_category_probabilities)
            )
            
            category = rng.choice(
                a=categories,
                p=customer_category_probabilities
            )

            priority = rng.choice(
                a=priority_options,
                p=priority_probabilities
            )

            sla_hours = priority_sla[priority] * category_sla_multiplier[category]

            resolution_multiplier = {
                "Billing": 0.85,
                "Account Management": 0.90,
                "Product Question": 0.95,
                "Technical Issue": 1.25,
                "Integration": 1.35,
                "Data Issue": 1.40
            }
        
            resolution_hours = round(
                sla_hours
                * resolution_multiplier[category]
                * rng.lognormal(mean=-0.15, sigma=0.60),
                2
            )

            expected_resolution = (created_time + dt.timedelta(hours=resolution_hours))

            ticket_unresolved_probability = min(
                unresolved_probability[category]
                * support_propensity,
                0.35
            )

            intentionally_unresolved = (
                rng.random() < ticket_unresolved_probability
            )

            if intentionally_unresolved:
                unresolved_delay_hours = rng.uniform(72, 168)
            
                delayed_resolution = (
                    created_time
                    + dt.timedelta(hours=unresolved_delay_hours)
                )
            
                if delayed_resolution >= pd.Timestamp(observation_end):
                    resolved_at = None
                    resolution_time = None
                    ticket_status = rng.choice(
                        ["Open", "In Progress"],
                        p=[0.65, 0.35]
                    )
                else:
                    resolved_at = delayed_resolution
                    resolution_time = round(unresolved_delay_hours, 2)
                    ticket_status = "Resolved"
            
            elif expected_resolution >= pd.Timestamp(observation_end):
                resolved_at = None
                resolution_time = None
                ticket_status = rng.choice(
                    ["Open", "In Progress"],
                    p=[0.65, 0.35]
                )
            
            else:
                resolved_at = expected_resolution
                resolution_time = resolution_hours
                ticket_status = "Resolved"
            
            if resolved_at is None:
                elapsed_hours = (
                    pd.Timestamp(observation_end) - created_time
                ).total_seconds() / 3600
            
                sla_breached = elapsed_hours > sla_hours
            else:
                sla_breached = resolution_time > sla_hours

            ticket_ids.append(fake.uuid4())
            customer_ids.append(customer_id)
            created_timestamps.append(created_time)
            resolved_timestamps.append(resolved_at)
            issue_categories.append(category)
            priorities.append(priority)
            ticket_statuses.append(ticket_status)
            resolution_times.append(resolution_time)
            sla_hours_values.append(round(sla_hours, 2))
            sla_breached_values.append(bool(sla_breached))
            

            
    support_tickets = pd.DataFrame({
        "ticket_id": ticket_ids,
        "customer_id": customer_ids,
        "created_at": created_timestamps,
        "resolved_at": resolved_timestamps,
        "issue_category": issue_categories,
        "priority": priorities,
        "ticket_status": ticket_statuses,
        "resolution_time_hours": resolution_times,
        "sla_hours": sla_hours_values,
        "sla_breached": sla_breached_values
    })

    support_tickets = support_tickets.sort_values(
        by=["customer_id", "created_at"]
    ).reset_index(drop=True)

    return support_tickets


In [36]:
support_tickets = create_support_tickets(customers, subscriptions, 42)

In [37]:
support_tickets.head()

,ticket_id,customer_id,created_at,resolved_at,issue_category,priority,ticket_status,resolution_time_hours,sla_hours,sla_breached
0,4223623b-cc3e-4dde-9ad5-cf06364d7c87,004b6fab-fcf5-4188-932e-6dcd83bc9478,2026-06-11 09:03:00,2026-06-12 11:01:48,Integration,Medium,Resolved,25.98,64.8,False
1,a255e1bb-0795-4c1f-938d-6b8cc607b1e9,00a85473-8242-4912-bda3-c1623a78b64a,2025-04-19 12:57:00,2025-04-23 12:48:36,Integration,Low,Resolved,95.86,97.2,False
2,9d264599-c4dd-4df5-9aec-56a857311609,00a85473-8242-4912-bda3-c1623a78b64a,2026-05-25 08:39:00,2026-05-26 02:37:12,Data Issue,High,Resolved,17.97,36.0,False
3,735741e5-fe1a-4ba9-8896-9c5b45f8c28d,00a85473-8242-4912-bda3-c1623a78b64a,2026-07-05 18:24:00,2026-07-08 02:17:24,Product Question,Low,Resolved,55.89,64.8,False
4,f80b2cea-658e-44a2-9879-cf35f4719986,00a85473-8242-4912-bda3-c1623a78b64a,2026-07-31 11:32:00,NaT,Technical Issue,Medium,Open,NaN,57.6,False


In [38]:
support_tickets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1216 entries, 0 to 1215
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ticket_id              1216 non-null   object        
 1   customer_id            1216 non-null   object        
 2   created_at             1216 non-null   datetime64[ns]
 3   resolved_at            1202 non-null   datetime64[ns]
 4   issue_category         1216 non-null   object        
 5   priority               1216 non-null   object        
 6   ticket_status          1216 non-null   object        
 7   resolution_time_hours  1202 non-null   float64       
 8   sla_hours              1216 non-null   float64       
 9   sla_breached           1216 non-null   bool          
dtypes: bool(1), datetime64[ns](2), float64(2), object(5)
memory usage: 86.8+ KB


In [39]:
tickets_by_cat = support_tickets.groupby("issue_category").agg(
    count_by_cat=("issue_category", "count")).reset_index()

In [40]:
tickets_by_cat

,issue_category,count_by_cat
0,Account Management,158
1,Billing,170
2,Data Issue,136
3,Integration,227
4,Product Question,213
5,Technical Issue,312


In [41]:
time_by_cat = support_tickets.groupby("issue_category").agg(
    by_cat=("resolution_time_hours", "mean")).reset_index()

In [42]:
time_by_cat

,issue_category,by_cat
0,Account Management,37.302293
1,Billing,32.556310
2,Data Issue,99.591544
3,Integration,88.569865
4,Product Question,45.639762
5,Technical Issue,82.862338


In [43]:
status_by_priority = support_tickets.groupby("priority").agg(
    statusby=("priority", "count")).reset_index()

In [44]:
status_by_priority

,priority,statusby
0,Critical,61
1,High,233
2,Low,322
3,Medium,600


In [45]:
support_tickets.describe()

,created_at,resolved_at,resolution_time_hours,sla_hours
count,1216,1202,1202.000000,1216.000000
mean,2025-12-02 03:58:42.286184192,2025-12-02 03:11:50.476428288,66.328952,52.717599
min,2024-08-27 08:15:00,2024-09-01 04:33:36,2.250000,9.000000
25%,2025-07-26 05:16:00,2025-07-26 03:30:45,25.557500,36.000000
50%,2026-01-07 16:22:30,2026-01-07 21:31:30,48.465000,57.600000
75%,2026-05-09 00:57:15,2026-05-07 14:04:54,91.657500,64.800000
max,2026-07-31 19:37:00,2026-07-31 23:51:12,492.300000,108.000000
std,NaN,NaN,57.426745,23.705377


In [46]:
support_tickets["issue_category"].value_counts(
    normalize=True
)

issue_category
Technical Issue       0.256579
Integration           0.186678
Product Question      0.175164
Billing               0.139803
Account Management    0.129934
Data Issue            0.111842
Name: proportion, dtype: float64

In [47]:
support_tickets["priority"].value_counts(
    normalize=True
)

priority
Medium      0.493421
Low         0.264803
High        0.191612
Critical    0.050164
Name: proportion, dtype: float64

In [48]:
support_tickets["ticket_status"].value_counts(
    normalize=True
)

ticket_status
Resolved       0.988487
Open           0.008224
In Progress    0.003289
Name: proportion, dtype: float64

In [49]:
support_tickets["ticket_id"].is_unique

True

In [50]:
support_tickets.isna().sum()

ticket_id                 0
customer_id               0
created_at                0
resolved_at              14
issue_category            0
priority                  0
ticket_status             0
resolution_time_hours    14
sla_hours                 0
sla_breached              0
dtype: int64

In [51]:
support_tickets["customer_id"].isin(customers["customer_id"]).all()

True

In [52]:
def finalize_subscriptions(customers, subscriptions, invoices, support_tickets, seed: int):
    rng = np.random.default_rng(seed)

    observation_end = dt.date(2026, 8, 1)

    early_churn_probabilities = {
        "Referral": 0.09,
        "Organic Search": 0.12,
        "Partner": 0.13,
        "Outbound Sales": 0.15,
        "Paid Search": 0.215,
        "Social Media": 0.245
    }

    late_churn_probability = 0.16

    customer_info = customers.set_index(
        "customer_id"
    ).to_dict("index")

    invoices_by_subscription = {
        subscription_id: group.sort_values("invoice_date")
        for subscription_id, group in invoices.groupby(
            "subscription_id"
        )
    }

    customer_ids_by_subscription = subscriptions.set_index(
        "subscription_id"
    )["customer_id"].to_dict()

    support_by_customer = {
        customer_id: group.sort_values("created_at")
        for customer_id, group in support_tickets.groupby(
            "customer_id"
        )
    }

    finalized = subscriptions.copy()

    end_dates = []
    statuses = []
    churn_types = []
    cancellation_reasons = []

    for _, subscription in finalized.iterrows():
        subscription_id = subscription["subscription_id"]
        customer_id = subscription["customer_id"]

        start_date = pd.to_datetime(
            subscription["start_date"]
        ).date()

        customer = customer_info[customer_id]
        channel = customer["acquisition_channel"]

        candidates = []

        # 1. Acquisition-related voluntary churn

        early_probability = early_churn_probabilities[channel]

        if rng.random() < early_probability:
            early_churn_date = start_date + dt.timedelta(
                days=int(rng.integers(15, 91))
            )

            if early_churn_date < observation_end:
                candidates.append((
                    early_churn_date,
                    "Voluntary",
                    "Acquisition / Poor Fit"
                ))

        else:
            # Background churn after the first 90 days.
            late_churn_date = start_date + dt.timedelta(
                days=int(rng.integers(91, 731))
            )

            if (
                rng.random() < late_churn_probability
                and late_churn_date < observation_end
            ):
                candidates.append((
                    late_churn_date,
                    "Voluntary",
                    "Other / Changed Requirements"
                ))

        # 2. Involuntary churn from exhausted payment retries

        customer_invoices = invoices_by_subscription.get(
            subscription_id,
            pd.DataFrame()
        )

        if not customer_invoices.empty:
            failed_invoices = customer_invoices[
                (customer_invoices["payment_status"] == "Failed")
                & (customer_invoices["retry_count"] >= 3)
            ]

            for _, invoice in failed_invoices.iterrows():
                last_attempt = invoice["last_attempt_date"]

                if pd.isna(last_attempt):
                    continue

                cancellation_date = (
                    pd.to_datetime(last_attempt).date()
                    + dt.timedelta(days=7)
                )

                if cancellation_date < observation_end:
                    candidates.append((
                        cancellation_date,
                        "Involuntary",
                        "Repeated Payment Failure"
                    ))

        # 3. Support-related voluntary churn

        customer_tickets = support_by_customer.get(
            customer_id,
            pd.DataFrame()
        )

        if not customer_tickets.empty:
            tickets = customer_tickets.copy()

            tickets["created_at"] = pd.to_datetime(
                tickets["created_at"]
            )

            signal_dates = {}

            # Signal 1: Repeated technical issues within 90 days

            technical_tickets = tickets[
                tickets["issue_category"] == "Technical Issue"
            ].sort_values("created_at")

            for i in range(len(technical_tickets)):
                first_date = technical_tickets.iloc[i]["created_at"]

                for j in range(i + 1, len(technical_tickets)):
                    second_date = technical_tickets.iloc[j]["created_at"]

                    if (second_date - first_date).days <= 90:
                        signal_dates["Repeated Technical Issues"] = (
                            second_date.date()
                        )
                        break

                if "Repeated Technical Issues" in signal_dates:
                    break

            # Signal 2: Repeated SLA breaches within 90 days

            breached_tickets = tickets[
                tickets["sla_breached"] == True
            ].sort_values("created_at")

            for i in range(len(breached_tickets)):
                first_date = breached_tickets.iloc[i]["created_at"]

                for j in range(i + 1, len(breached_tickets)):
                    second_date = breached_tickets.iloc[j]["created_at"]

                    if (second_date - first_date).days <= 90:
                        signal_dates["Repeated SLA Breaches"] = (
                            second_date.date()
                        )
                        break

                if "Repeated SLA Breaches" in signal_dates:
                    break

            # Signal 3: High/Critical ticket unresolved for 72+ hours

            high_priority_tickets = tickets[
                tickets["priority"].isin(["High", "Critical"])
                & tickets["ticket_status"].isin(["Open", "In Progress"])
                & tickets["resolved_at"].isna()
            ]

            for _, ticket in high_priority_tickets.iterrows():
                escalation_date = (
                    ticket["created_at"] + pd.Timedelta(hours=72)
                ).date()

                if escalation_date < observation_end:
                    signal_dates["Unresolved High-Priority Ticket"] = (
                        escalation_date
                    )
                    break

            # Determine risk as support problems accumulate

            active_signals = set()

            for signal_date, signal_name in sorted(
                (date, name)
                for name, date in signal_dates.items()
            ):
                active_signals.add(signal_name)

                signal_count = len(active_signals)

                support_probability = {
                    1: 0.25,
                    2: 0.55,
                    3: 0.65
                }[signal_count]

                if rng.random() < support_probability:
                    support_cancellation_date = (
                        signal_date
                        + dt.timedelta(
                            days=int(rng.integers(7, 31))
                        )
                    )

                    if support_cancellation_date < observation_end:
                        candidates.append((
                            support_cancellation_date,
                            "Voluntary",
                            "Poor Support Experience"
                        ))

                    break

        # 4. Select earliest eligible cancellation

        eligible_candidates = [
            candidate
            for candidate in candidates
            if candidate[0] > start_date
            and candidate[0] < observation_end
        ]

        if eligible_candidates:
            churn_date, churn_type, reason = min(
                eligible_candidates,
                key=lambda candidate: candidate[0]
            )

            end_dates.append(churn_date)
            statuses.append("Cancelled")
            churn_types.append(churn_type)
            cancellation_reasons.append(reason)

        else:
            end_dates.append(None)
            statuses.append("Active")
            churn_types.append(None)
            cancellation_reasons.append(None)

    finalized["end_date"] = end_dates
    finalized["subscription_status"] = statuses
    finalized["churn_type"] = churn_types
    finalized["cancellation_reason"] = cancellation_reasons

    # 5. Remove records created on or after cancellation

    end_date_lookup = finalized.set_index(
        "subscription_id"
    )["end_date"].to_dict()

    reconciled_invoices = invoices.copy()

    reconciled_invoices["subscription_end_date"] = (
        reconciled_invoices["subscription_id"].map(
            end_date_lookup
        )
    )

    reconciled_invoices = reconciled_invoices[
        reconciled_invoices.apply(
            lambda row: (
                pd.isna(row["subscription_end_date"])
                or row["invoice_date"] < row["subscription_end_date"]
            ),
            axis=1
        )
    ].drop(columns="subscription_end_date").reset_index(drop=True)

    reconciled_tickets = support_tickets.copy()

    reconciled_tickets["subscription_end_date"] = (
        reconciled_tickets["customer_id"].map(
            finalized.set_index("customer_id")["end_date"]
        )
    )

    reconciled_tickets = reconciled_tickets[
        reconciled_tickets.apply(
            lambda row: (
                pd.isna(row["subscription_end_date"])
                or row["created_at"].date() < row["subscription_end_date"]
            ),
            axis=1
        )
    ].drop(columns="subscription_end_date").reset_index(drop=True)

    return finalized, reconciled_invoices, reconciled_tickets

In [53]:
customers = create_customer_data(n_rows=5000, seed=42)

In [54]:
subscriptions = create_subscription_data(customers=customers, seed=43)

In [55]:
subscription_events = create_subscription_events(subscriptions=subscriptions, seed=44)

In [56]:
invoices = create_invoice_data(customers=customers, subscriptions=subscriptions, 
                               subscription_events=subscription_events, 
                               seed=45)

In [57]:
support_tickets = create_support_tickets(customers=customers, subscriptions=subscriptions, seed=46)

In [58]:
subscriptions, invoices, support_tickets = finalize_subscriptions(
    customers=customers, subscriptions=subscriptions, 
    invoices=invoices, support_tickets=support_tickets, 
    seed=47)

In [59]:
subscription_events = subscription_events.merge(
    subscriptions[[
        "subscription_id",
        "end_date"
    ]],
    on="subscription_id",
    how="left"
)

subscription_events = subscription_events[
    subscription_events["end_date"].isna()
    | (
        subscription_events["event_date"]
        < subscription_events["end_date"]
    )
].drop(
    columns="end_date"
).reset_index(drop=True)

In [60]:
usage_events = create_usage_events(customers=customers, subscriptions=subscriptions,seed=48)

In [61]:
print(subscriptions["subscription_status"].value_counts())

print(subscriptions["churn_type"].value_counts(dropna=False))

print(
    subscriptions.merge(right=customers[["customer_id", "acquisition_channel"]],
        on="customer_id"
    )
    .groupby("acquisition_channel")["subscription_status"]
    .value_counts(normalize=True)
)

invoice_analysis = invoices.merge(right=subscriptions[["subscription_id", "customer_id", "billing_cycle"]],
    on="subscription_id"
).merge(right=customers[["customer_id", "country"]],
    on="customer_id"
)

print(
    invoice_analysis.groupby(
        ["country", "billing_cycle", "payment_status"]
    ).size()
)

# Support performance
print(support_tickets["ticket_status"].value_counts())
print(support_tickets["sla_breached"].value_counts())

# Dataset sizes
for name, df in {
    "customers": customers,
    "subscriptions": subscriptions,
    "subscription_events": subscription_events,
    "invoices": invoices,
    "support_tickets": support_tickets,
    "usage_events": usage_events
}.items():
    print(f"{name}: {len(df):,} rows")

subscription_status
Active       3654
Cancelled    1346
Name: count, dtype: int64
churn_type
None           3654
Voluntary      1047
Involuntary     299
Name: count, dtype: int64
acquisition_channel  subscription_status
Organic Search       Active                 0.765186
                     Cancelled              0.234814
Outbound Sales       Active                 0.747967
                     Cancelled              0.252033
Paid Search          Active                 0.692970
                     Cancelled              0.307030
Partner              Active                 0.771242
                     Cancelled              0.228758
Referral             Active                 0.765492
                     Cancelled              0.234508
Social Media         Active                 0.649306
                     Cancelled              0.350694
Name: proportion, dtype: float64
country    billing_cycle  payment_status
Australia  Annual         Failed                1
                    

In [62]:
payment_analysis = invoices.merge(
    subscriptions[[
        "subscription_id",
        "customer_id",
        "billing_cycle"
    ]],
    on="subscription_id"
).merge(
    customers[[
        "customer_id",
        "country"
    ]],
    on="customer_id"
)

payment_analysis["initial_failure"] = (
    payment_analysis["failure_reason"].notna()
)

payment_analysis["assessable"] = (
    payment_analysis["payment_status"] != "Pending"
)

print(
    payment_analysis[
        payment_analysis["assessable"]
    ].groupby(
        ["country", "billing_cycle"]
    ).agg(
        invoices=("invoice_id", "count"),
        initial_failures=("initial_failure", "sum"),
        initial_failure_rate=("initial_failure", "mean"),
        recovered_payments=("recovered", "sum")
    )
)

                         invoices  initial_failures  initial_failure_rate  \
country   billing_cycle                                                     
Australia Annual              140                 8              0.057143   
          Monthly            2055                90              0.043796   
Canada    Annual              338                 7              0.020710   
          Monthly            5377               206              0.038311   
France    Annual              181                 4              0.022099   
          Monthly            2804               132              0.047076   
Germany   Annual              202                 4              0.019802   
          Monthly            3425               116              0.033869   
UK        Annual              253                 7              0.027668   
          Monthly            4058               286              0.070478   
US        Annual              788                17              0.021574   

In [63]:
print(
    support_tickets.groupby(
        ["ticket_status", "sla_breached"]
    ).size().unstack(fill_value=0)
)

print(
    support_tickets.groupby(
        "issue_category"
    ).agg(
        tickets=("ticket_id", "count"),
        sla_breach_rate=("sla_breached", "mean"),
        avg_resolution_hours=("resolution_time_hours", "mean")
    )
)

sla_breached   False  True 
ticket_status              
In Progress       12      3
Open              12      8
Resolved        2311   2550
                    tickets  sla_breach_rate  avg_resolution_hours
issue_category                                                    
Account Management      570         0.392982             42.170684
Billing                 725         0.351724             35.258033
Data Issue              563         0.655417            100.943399
Integration             932         0.673820             94.464500
Product Question        879         0.392491             44.497082
Technical Issue        1227         0.603097             79.151682


In [64]:
# Cancellation reasons
print(
    subscriptions["cancellation_reason"]
    .value_counts(dropna=False)
)

# Mature customer cohorts
subscription_analysis = subscriptions.merge(
    customers[[
        "customer_id",
        "acquisition_channel",
        "signup_date"
    ]],
    on="customer_id"
)

subscription_analysis["signup_date"] = pd.to_datetime(
    subscription_analysis["signup_date"]
)

subscription_analysis["end_date"] = pd.to_datetime(
    subscription_analysis["end_date"]
)

observation_end = pd.Timestamp("2026-08-01")

mature_customers = subscription_analysis[
    subscription_analysis["signup_date"]
    <= observation_end - pd.Timedelta(days=90)
].copy()

mature_customers["churned_within_90_days"] = (
    mature_customers["end_date"].notna()
    & (
        mature_customers["end_date"]
        <= mature_customers["signup_date"]
        + pd.Timedelta(days=90)
    )
)

print(
    mature_customers.groupby(
        "acquisition_channel"
    )["churned_within_90_days"].mean().sort_values(
        ascending=False
    )
)

cancellation_reason
None                            3654
Acquisition / Poor Fit           694
Repeated Payment Failure         299
Other / Changed Requirements     256
Poor Support Experience           97
Name: count, dtype: int64
acquisition_channel
Social Media      0.246124
Paid Search       0.227014
Outbound Sales    0.153670
Partner           0.141791
Organic Search    0.140206
Referral          0.114641
Name: churned_within_90_days, dtype: float64


In [65]:
support_analysis = support_tickets[
    support_tickets["sla_breached"]
].copy()

support_analysis["created_at"] = pd.to_datetime(
    support_analysis["created_at"]
)

support_counts = []

for customer_id, group in support_analysis.groupby("customer_id"):
    dates = sorted(group["created_at"])

    qualifies = any(
        (dates[j] - dates[i]).days <= 90
        for i in range(len(dates))
        for j in range(i + 1, len(dates))
    )

    if qualifies:
        support_counts.append(customer_id)

print("Customers with at least two SLA breaches within 90 days:",
      len(support_counts))

Customers with at least two SLA breaches within 90 days: 313


In [66]:
technical_tickets = support_tickets[
    (support_tickets["issue_category"] == "Technical Issue") &
    (support_tickets["ticket_status"].isin(["Open", "In Progress"]))
].copy()

technical_tickets["created_at"] = pd.to_datetime(
    technical_tickets["created_at"]
)

qualifying_customers = []

for customer_id, group in technical_tickets.groupby("customer_id"):
    dates = sorted(group["created_at"])

    qualifies = any(
        (dates[j] - dates[i]).days <= 90
        for i in range(len(dates))
        for j in range(i + 1, len(dates))
    )

    if qualifies:
        qualifying_customers.append(customer_id)

print(
    "Customers with at least two unresolved technical issues within 90 days:",
    len(qualifying_customers)
)

Customers with at least two unresolved technical issues within 90 days: 0


In [67]:
observation_end = pd.Timestamp("2026-08-01")

unresolved = support_tickets[
    support_tickets["ticket_status"].isin(["Open", "In Progress"])
].copy()

unresolved["age_hours"] = (
    observation_end - pd.to_datetime(unresolved["created_at"])
).dt.total_seconds() / 3600

print(
    unresolved.groupby("priority")["age_hours"]
    .agg(["count", "mean", "median", "max"])
)

print(
    unresolved.groupby("priority")["age_hours"]
    .apply(lambda x: (x >= 72).sum())
)

          count       mean     median         max
priority                                         
Critical      1   6.916667   6.916667    6.916667
High          5  50.313333  51.766667   78.483333
Low          13  59.457692  51.483333  171.766667
Medium       16  44.363542  31.575000  123.316667
priority
Critical    0
High        2
Low         4
Medium      4
Name: age_hours, dtype: int64


In [68]:
import pandas as pd

observation_end = pd.Timestamp("2026-08-01")

# ---------------------------------------------------------------------------
# Base views
# ---------------------------------------------------------------------------
subs = subscriptions.merge(
    customers[["customer_id", "acquisition_channel", "signup_date"]],
    on="customer_id", how="left"
)
subs["start_date"] = pd.to_datetime(subs["start_date"])
subs["end_date"]   = pd.to_datetime(subs["end_date"])
subs["is_churned"] = subs["subscription_status"].eq("Cancelled")

tickets = support_tickets.copy()
tickets["created_at"]  = pd.to_datetime(tickets["created_at"])
tickets["resolved_at"] = pd.to_datetime(tickets["resolved_at"])

# ---------------------------------------------------------------------------
# 1. Per-customer support-condition flags (mirror finalize_subscriptions logic)
# ---------------------------------------------------------------------------
def customers_with_repeated_breaches(tickets, window_days=90):
    """Customers with >=2 SLA-breached tickets within window_days of each other."""
    breached = tickets[tickets["sla_breached"]].sort_values(["customer_id", "created_at"])
    hits = set()
    for cid, grp in breached.groupby("customer_id"):
        dates = grp["created_at"].tolist()
        for i in range(len(dates)):
            for j in range(i + 1, len(dates)):
                if (dates[j] - dates[i]).days <= window_days:
                    hits.add(cid)
                    break
            if cid in hits:
                break
    return hits

def customers_with_repeated_technical(tickets, window_days=90):
    """Customers with >=2 Technical Issue tickets within window_days."""
    tech = tickets[tickets["issue_category"].eq("Technical Issue")] \
             .sort_values(["customer_id", "created_at"])
    hits = set()
    for cid, grp in tech.groupby("customer_id"):
        dates = grp["created_at"].tolist()
        for i in range(len(dates)):
            for j in range(i + 1, len(dates)):
                if (dates[j] - dates[i]).days <= window_days:
                    hits.add(cid)
                    break
            if cid in hits:
                break
    return hits

def customers_with_stale_high_priority(tickets, hours=72):
    """Customers with a High/Critical ticket unresolved for >= hours at observation_end."""
    open_high = tickets[
        tickets["priority"].isin(["High", "Critical"])
        & tickets["ticket_status"].isin(["Open", "In Progress"])
    ].copy()
    open_high["age_hours"] = (observation_end - open_high["created_at"]).dt.total_seconds() / 3600
    return set(open_high.loc[open_high["age_hours"] >= hours, "customer_id"].unique())

repeated_breach_cids = customers_with_repeated_breaches(tickets)
repeated_tech_cids   = customers_with_repeated_technical(tickets)
stale_high_cids      = customers_with_stale_high_priority(tickets)

condition_flags = pd.DataFrame({
    "customer_id": subs["customer_id"],
})
condition_flags["repeated_breaches"]      = condition_flags["customer_id"].isin(repeated_breach_cids)
condition_flags["repeated_technical"]     = condition_flags["customer_id"].isin(repeated_tech_cids)
condition_flags["stale_high_priority"]    = condition_flags["customer_id"].isin(stale_high_cids)
condition_flags["any_support_signal"]     = condition_flags[["repeated_breaches",
                                                            "repeated_technical",
                                                            "stale_high_priority"]].any(axis=1)
condition_flags["support_signal_count"]   = condition_flags[["repeated_breaches",
                                                            "repeated_technical",
                                                            "stale_high_priority"]].sum(axis=1)

# ---------------------------------------------------------------------------
# Q1. Number of customers meeting each support condition
# ---------------------------------------------------------------------------
print("=" * 60)
print("Q1: CUSTOMERS MEETING EACH SUPPORT CONDITION")
print("=" * 60)
counts = condition_flags[["repeated_breaches", "repeated_technical",
                          "stale_high_priority", "any_support_signal"]].sum()
print(counts.rename("customers_meeting_condition"))
print()
print("Signal-count distribution (how many signals per customer):")
print(condition_flags["support_signal_count"].value_counts().sort_index())
print()

# ---------------------------------------------------------------------------
# Q2. Number of support-related cancellations
# ---------------------------------------------------------------------------
print("=" * 60)
print("Q2: SUPPORT-RELATED CANCELLATIONS")
print("=" * 60)
reason_counts = subs["cancellation_reason"].value_counts(dropna=False)
print(reason_counts)
print()
support_cancels = subs[subs["cancellation_reason"].eq("Poor Support Experience")]
print(f"Support-related cancellations: {len(support_cancels):,} "
      f"({len(support_cancels)/max(len(subs),1):.2%} of all subs, "
      f"{len(support_cancels)/max(int(subs['is_churned'].sum()),1):.2%} of churned subs)")
print()
# Cross-check: support-driven cancellations should come from flagged customers
support_cancel_flagged = support_cancels.merge(
    condition_flags[["customer_id", "any_support_signal", "support_signal_count"]],
    on="customer_id", how="left"
)
print("Support cancellations whose customer has a support signal flag: "
      f"{support_cancel_flagged['any_support_signal'].sum()} / {len(support_cancel_flagged)}")
print()

# ---------------------------------------------------------------------------
# Q3. Churn rate: support-flagged vs. not
# ---------------------------------------------------------------------------
print("=" * 60)
print("Q3: CHURN RATE FOR CUSTOMERS WITH vs. WITHOUT SUPPORT ISSUES")
print("=" * 60)

analysis = subs.merge(condition_flags, on="customer_id", how="left")

# Overall
overall = analysis.groupby("any_support_signal").agg(
    customers=("customer_id", "count"),
    churned=("is_churned", "sum"),
    churn_rate=("is_churned", "mean"),
)
print("By any-support-signal:")
print(overall)
print()

# By individual signal
for col in ["repeated_breaches", "repeated_technical", "stale_high_priority"]:
    print(f"By {col}:")
    print(analysis.groupby(col).agg(
        customers=("customer_id", "count"),
        churned=("is_churned", "sum"),
        churn_rate=("is_churned", "mean"),
    ))
    print()

# By number of signals (dose-response check)
print("By signal count:")
print(analysis.groupby("support_signal_count").agg(
    customers=("customer_id", "count"),
    churned=("is_churned", "sum"),
    churn_rate=("is_churned", "mean"),
))
print()

# Isolation check: support signal vs. churn EXCLUDING acquisition-driven churn,
# to confirm the finding isn't just "risky channels have more tickets"
print("Churn rate by support signal, holding channel fixed (top channels):")
top_channels = subs["acquisition_channel"].value_counts().head(4).index
print(
    analysis[analysis["acquisition_channel"].isin(top_channels)]
    .groupby(["acquisition_channel", "any_support_signal"])
    .agg(customers=("customer_id", "count"),
         churned=("is_churned", "sum"),
         churn_rate=("is_churned", "mean"))
)

Q1: CUSTOMERS MEETING EACH SUPPORT CONDITION
repeated_breaches      313
repeated_technical      97
stale_high_priority      2
any_support_signal     367
Name: customers_meeting_condition, dtype: int64

Signal-count distribution (how many signals per customer):
support_signal_count
0    4633
1     322
2      45
Name: count, dtype: int64

Q2: SUPPORT-RELATED CANCELLATIONS
cancellation_reason
None                            3654
Acquisition / Poor Fit           694
Repeated Payment Failure         299
Other / Changed Requirements     256
Poor Support Experience           97
Name: count, dtype: int64

Support-related cancellations: 97 (1.94% of all subs, 7.21% of churned subs)

Support cancellations whose customer has a support signal flag: 97 / 97

Q3: CHURN RATE FOR CUSTOMERS WITH vs. WITHOUT SUPPORT ISSUES
By any-support-signal:
                    customers  churned  churn_rate
any_support_signal                                
False                    4633     1212    0.261602
True   

In [69]:
# Get cancellation dates
cancellation_dates = (
    subscriptions.loc[
        subscriptions["subscription_status"] == "Cancelled",
        ["subscription_id", "customer_id", "end_date"]
    ]
)

# Join cancellation dates to each activity table
activity_tables = {
    "invoices": (invoices, "subscription_id"),
    "support_tickets": (support_tickets, "customer_id"),
    "usage_events": (usage_events, "customer_id"),
    "subscription_events": (subscription_events, "subscription_id")
}

for table_name, (df, join_column) in activity_tables.items():

    check = df.merge(
        cancellation_dates,
        on=join_column,
        how="inner"
    )

    if table_name == "invoices":
        activity_date = check["invoice_date"]
    elif table_name == "support_tickets":
        activity_date = check["created_at"]
    elif table_name == "usage_events":
        activity_date = check["event_timestamp"]
    else:
        activity_date = check["event_date"]

    violations = activity_date >= check["end_date"]

    print(f"{table_name}: {violations.sum()} post-cancellation records")

invoices: 0 post-cancellation records
support_tickets: 0 post-cancellation records
usage_events: 0 post-cancellation records
subscription_events: 0 post-cancellation records


In [70]:
print(
    "Active subscriptions with end dates:",
    (
        (subscriptions["subscription_status"] == "Active") &
        subscriptions["end_date"].notna()
    ).sum()
)

print(
    "Cancelled subscriptions without end dates:",
    (
        (subscriptions["subscription_status"] == "Cancelled") &
        subscriptions["end_date"].isna()
    ).sum()
)

Active subscriptions with end dates: 0
Cancelled subscriptions without end dates: 0


In [71]:
print(
    subscriptions.groupby(
        ["subscription_status", "cancellation_reason"],
        dropna=False
    ).size()
)

subscription_status  cancellation_reason         
Active               NaN                             3654
Cancelled            Acquisition / Poor Fit           694
                     Other / Changed Requirements     256
                     Poor Support Experience           97
                     Repeated Payment Failure         299
dtype: int64


In [72]:
event_check = subscription_events.merge(
    subscriptions[["subscription_id", "start_date", "end_date"]],
    on="subscription_id",
    how="left"
)

print(
    "Events on or before signup:",
    (event_check["event_date"] <= event_check["start_date"]).sum()
)

print(
    "Events on or after cancellation:",
    (
        event_check["end_date"].notna() &
        (event_check["event_date"] >= event_check["end_date"])
    ).sum()
)

Events on or before signup: 0
Events on or after cancellation: 0


In [73]:
# Churn rate by acquisition channel
print(
    subscriptions.merge(
        customers[["customer_id", "acquisition_channel"]],
        on="customer_id"
    )
    .groupby("acquisition_channel")
    .agg(
        customers=("subscription_id", "count"),
        churned=("end_date", "count"),
        churn_rate=("end_date", lambda x: x.notna().mean())
    )
    .sort_values("churn_rate", ascending=False)
    .round(3)
)

                     customers  churned  churn_rate
acquisition_channel                                
Social Media               576      202       0.351
Paid Search               1394      428       0.307
Outbound Sales             492      124       0.252
Organic Search            1103      259       0.235
Referral                   823      193       0.235
Partner                    612      140       0.229


In [74]:
# Billing cycle
print(subscriptions.groupby("billing_cycle").agg(
    customers=("subscription_id", "count"),
    churned=("end_date", "count"),
    churn_rate=("end_date", lambda x: x.notna().mean())
).round(3))

# Country
print(subscriptions.merge(
    customers[["customer_id", "country"]],
    on="customer_id"
).groupby("country").agg(
    customers=("subscription_id", "count"),
    churned=("end_date", "count"),
    churn_rate=("end_date", lambda x: x.notna().mean())
).round(3))

# Plan
print(subscriptions.groupby("plan").agg(
    customers=("subscription_id", "count"),
    churned=("end_date", "count"),
    churn_rate=("end_date", lambda x: x.notna().mean())
).round(3))

               customers  churned  churn_rate
billing_cycle                                
Annual              1439      327       0.227
Monthly             3561     1019       0.286
           customers  churned  churn_rate
country                                  
Australia        339       96       0.283
Canada           849      232       0.273
France           441      110       0.249
Germany          525      139       0.265
UK               641      180       0.281
US              2205      589       0.267
            customers  churned  churn_rate
plan                                      
Basic            1920      482       0.251
Enterprise        802      243       0.303
Pro              2278      621       0.273


In [75]:
plan_channel = subscriptions.merge(
    customers[["customer_id", "acquisition_channel"]],
    on="customer_id"
)

print(
    plan_channel.groupby(
        ["acquisition_channel", "plan"]
    ).agg(
        customers=("subscription_id", "count"),
        churned=("end_date", "count"),
        churn_rate=("end_date", lambda x: x.notna().mean())
    ).round(3)
)

                                customers  churned  churn_rate
acquisition_channel plan                                      
Organic Search      Basic             449       92       0.205
                    Enterprise        171       41       0.240
                    Pro               483      126       0.261
Outbound Sales      Basic             194       49       0.253
                    Enterprise         78       22       0.282
                    Pro               220       53       0.241
Paid Search         Basic             532      155       0.291
                    Enterprise        206       67       0.325
                    Pro               656      206       0.314
Partner             Basic             223       49       0.220
                    Enterprise        118       33       0.280
                    Pro               271       58       0.214
Referral            Basic             309       68       0.220
                    Enterprise        134       40     

In [76]:
support_plan = (
    subscriptions.merge(
        support_tickets.groupby("customer_id").agg(
            ticket_count=("ticket_id", "count"),
            sla_breaches=("sla_breached", "sum")
        ),
        on="customer_id",
        how="left"
    )
)

support_plan[["ticket_count", "sla_breaches"]] = (
    support_plan[["ticket_count", "sla_breaches"]].fillna(0)
)

print(
    support_plan.groupby("plan").agg(
        customers=("subscription_id", "count"),
        churn_rate=("end_date", lambda x: x.notna().mean()),
        avg_tickets=("ticket_count", "mean"),
        avg_breaches=("sla_breaches", "mean")
    ).round(3)
)

            customers  churn_rate  avg_tickets  avg_breaches
plan                                                        
Basic            1920       0.251        0.677         0.365
Enterprise        802       0.303        1.520         0.774
Pro              2278       0.273        1.043         0.544


In [77]:
support_flags = (
    support_tickets.groupby("customer_id")
    .agg(
        repeated_breaches=("sla_breached", "sum"),
        repeated_technical=(
            "issue_category",
            lambda x: (x == "Technical Issue").sum()
        )
    )
    .reset_index()
)

support_flags["any_support_signal"] = (
    (support_flags["repeated_breaches"] >= 2) |
    (support_flags["repeated_technical"] >= 2)
)



plan_support = subscriptions.merge(
    support_flags[["customer_id", "any_support_signal"]],
    on="customer_id",
    how="left"
)

plan_support["any_support_signal"] = (
    plan_support["any_support_signal"].fillna(False)
)

print(
    plan_support.groupby(["plan", "any_support_signal"]).agg(
        customers=("subscription_id", "count"),
        churned=("end_date", "count"),
        churn_rate=("end_date", lambda x: x.notna().mean())
    ).round(3)
)

                               customers  churned  churn_rate
plan       any_support_signal                                
Basic      False                    1766      445       0.252
           True                      154       37       0.240
Enterprise False                     619      194       0.313
           True                      183       49       0.268
Pro        False                    1962      556       0.283
           True                      316       65       0.206


/var/folders/81/k7y0g_pd0qbg39315nz9j_7h0000gn/T/ipykernel_32962/2275509724.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  plan_support["any_support_signal"].fillna(False)


In [78]:
print(
    subscriptions.groupby(
        ["plan", "cancellation_reason"],
        dropna=False
    ).size().unstack(fill_value=0)
)

cancellation_reason  Acquisition / Poor Fit  Other / Changed Requirements  \
plan                                                                        
Basic                                   250                           100   
Enterprise                              122                            40   
Pro                                     322                           116   

cancellation_reason  Poor Support Experience  Repeated Payment Failure   NaN  
plan                                                                          
Basic                                     22                       110  1438  
Enterprise                                33                        48   559  
Pro                                       42                       141  1657  


In [79]:
print(
    subscriptions.assign(
        support_churned=(
            subscriptions["cancellation_reason"]
            == "Poor Support Experience"
        )
    )
    .groupby("plan")
    .agg(
        customers=("subscription_id", "count"),
        support_cancellations=("support_churned", "sum"),
        support_cancellation_rate=("support_churned", "mean")
    )
    .round(3)
)

            customers  support_cancellations  support_cancellation_rate
plan                                                                   
Basic            1920                     22                      0.011
Enterprise        802                     33                      0.041
Pro              2278                     42                      0.018


In [80]:
payment_analysis = subscriptions.merge(
    customers[["customer_id", "country"]],
    on="customer_id"
)

payment_analysis["payment_churned"] = (
    payment_analysis["cancellation_reason"]
    == "Repeated Payment Failure"
)

print(
    payment_analysis.groupby(["country", "billing_cycle"]).agg(
        customers=("subscription_id", "count"),
        payment_cancellations=("payment_churned", "sum"),
        payment_churn_rate=("payment_churned", "mean")
    ).round(4)
)

                         customers  payment_cancellations  payment_churn_rate
country   billing_cycle                                                      
Australia Annual               109                      1              0.0092
          Monthly              230                     14              0.0609
Canada    Annual               251                      0              0.0000
          Monthly              598                     48              0.0803
France    Annual               136                      0              0.0000
          Monthly              305                     29              0.0951
Germany   Annual               154                      1              0.0065
          Monthly              371                     21              0.0566
UK        Annual               194                      1              0.0052
          Monthly              447                     45              0.1007
US        Annual               595                      5       

In [81]:
payment_analysis["segment"] = np.where(
    (payment_analysis["country"] == "UK") &
    (payment_analysis["billing_cycle"] == "Monthly"),
    "UK Monthly",
    "All Other Segments"
)

print(
    payment_analysis.groupby("segment").agg(
        customers=("subscription_id", "count"),
        payment_cancellations=("payment_churned", "sum"),
        payment_churn_rate=("payment_churned", "mean")
    ).round(4)
)

                    customers  payment_cancellations  payment_churn_rate
segment                                                                 
All Other Segments       4553                    254              0.0558
UK Monthly                447                     45              0.1007


In [82]:
print("Subscriptions:")
print(subscriptions.columns.tolist())

print("\nSubscription events:")
print(subscription_events.columns.tolist())

Subscriptions:
['subscription_id', 'customer_id', 'start_date', 'end_date', 'subscription_status', 'billing_cycle', 'plan', 'monthly_price', 'churn_type', 'cancellation_reason']

Subscription events:
['event_id', 'subscription_id', 'event_date', 'event_type', 'previous_plan', 'new_plan', 'previous_price', 'new_price']


In [83]:
print(subscriptions.head(3).to_string())
print(subscription_events.head(3).to_string())

                        subscription_id                           customer_id  start_date    end_date subscription_status billing_cycle   plan  monthly_price churn_type     cancellation_reason
0  c33f4584-b23b-41d8-893c-d01609de8895  bdd640fb-0667-4ad1-9c80-317fa3b1799d  2025-02-17        None              Active       Monthly    Pro          79.00       None                    None
1  5eb04521-7668-4387-b59b-a79924d8cea5  1a3d1fa7-bc89-40a9-a3b8-c1e9392456de  2025-12-07  2026-02-17           Cancelled        Annual  Basic          24.65  Voluntary  Acquisition / Poor Fit
2  18a61865-cafe-4acf-b2cc-dfa7abf10ac2  972a8469-1641-4f82-8b9d-2434e465e150  2025-06-04        None              Active       Monthly    Pro          79.00       None                    None
                               event_id                       subscription_id  event_date event_type previous_plan    new_plan  previous_price  new_price
0  b39cfd4b-8abe-4d78-8520-10116895cea8  9be6a8ea-7f8e-44c2-994b-7a5674043

In [84]:
# Get the latest plan change for each subscription
latest_events = (
    subscription_events
    .sort_values("event_date")
    .drop_duplicates("subscription_id", keep="last")
    [["subscription_id", "new_plan", "new_price"]]
    .rename(columns={
        "new_plan": "effective_plan",
        "new_price": "effective_monthly_price"
    })
)

# Merge with subscriptions
arr_analysis = subscriptions.merge(
    latest_events,
    on="subscription_id",
    how="left"
)

# Use original plan and price when no plan changes occurred
arr_analysis["effective_plan"] = (
    arr_analysis["effective_plan"].fillna(arr_analysis["plan"])
)

arr_analysis["effective_monthly_price"] = (
    arr_analysis["effective_monthly_price"]
    .fillna(arr_analysis["monthly_price"])
)

# Calculate ARR
arr_analysis["lost_arr"] = (
    arr_analysis["effective_monthly_price"] * 12
)

In [85]:
payment_losses = arr_analysis[
    arr_analysis["cancellation_reason"] == "Repeated Payment Failure"
]

print(
    payment_losses.agg(
        payment_cancellations=("subscription_id", "count"),
        total_lost_arr=("lost_arr", "sum"),
        average_lost_arr=("lost_arr", "mean")
    ).round(2)
)

                       subscription_id   lost_arr
payment_cancellations            299.0        NaN
total_lost_arr                     NaN  316172.40
average_lost_arr                   NaN    1057.43


In [86]:
arr_analysis = arr_analysis.merge(
    customers[["customer_id", "country"]],
    on="customer_id",
    how="left"
)

payment_losses = arr_analysis[
    arr_analysis["cancellation_reason"] == "Repeated Payment Failure"
]

payment_losses["segment"] = np.where(
    (payment_losses["country"] == "UK") &
    (payment_losses["billing_cycle"] == "Monthly"),
    "UK Monthly",
    "All Other Segments"
)

print(
    payment_losses.groupby("segment").agg(
        cancellations=("subscription_id", "count"),
        lost_arr=("lost_arr", "sum"),
        average_lost_arr=("lost_arr", "mean")
    ).round(2)
)

/var/folders/81/k7y0g_pd0qbg39315nz9j_7h0000gn/T/ipykernel_32962/3990381752.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  payment_losses["segment"] = np.where(


                    cancellations  lost_arr  average_lost_arr
segment                                                      
All Other Segments            254  264872.4            1042.8
UK Monthly                     45   51300.0            1140.0


In [87]:
reason_impact = (
    arr_analysis.groupby("cancellation_reason", dropna=False)
    .agg(
        cancellations=("subscription_id", "count"),
        lost_arr=("lost_arr", "sum"),
        average_lost_arr=("lost_arr", "mean")
    )
    .round(2)
)

print(reason_impact)

                              cancellations   lost_arr  average_lost_arr
cancellation_reason                                                     
Acquisition / Poor Fit                  694   661390.8            953.01
Other / Changed Requirements            256   263020.8           1027.42
Poor Support Experience                  97   123072.6           1268.79
Repeated Payment Failure                299   316172.4           1057.43
NaN                                    3654  3817449.6           1044.73


In [88]:
channel_arr = arr_analysis.merge(
    customers[["customer_id", "acquisition_channel"]],
    on="customer_id",
    how="left"
)

# Keep active subscriptions
active_subscriptions = channel_arr[
    channel_arr["subscription_status"] == "Active"
]

# Summarize by acquisition channel
retained_arr = (
    active_subscriptions.groupby("acquisition_channel")
    .agg(
        active_customers=("subscription_id", "count"),
        retained_arr=("lost_arr", "sum"),
        avg_arr_per_customer=("lost_arr", "mean")
    )
)

# Add total acquired customers and retention rate
total_customers = customers.groupby("acquisition_channel").size()

retained_arr["total_customers"] = total_customers
retained_arr["retention_rate"] = (
    retained_arr["active_customers"] /
    retained_arr["total_customers"]
)

# Calculate retained ARR per originally acquired customer
retained_arr["retained_arr_per_acquired_customer"] = (
    retained_arr["retained_arr"] /
    retained_arr["total_customers"]
)

print(
    retained_arr.sort_values(
        "retained_arr_per_acquired_customer",
        ascending=False
    ).round(2)
)

                     active_customers  retained_arr  avg_arr_per_customer  \
acquisition_channel                                                         
Partner                           472      513242.4               1087.38   
Outbound Sales                    368      399441.0               1085.44   
Referral                          630      646804.8               1026.67   
Organic Search                    844      865196.4               1025.11   
Paid Search                       966      989883.0               1024.72   
Social Media                      374      402882.0               1077.22   

                     total_customers  retention_rate  \
acquisition_channel                                    
Partner                          612            0.77   
Outbound Sales                   492            0.75   
Referral                         823            0.77   
Organic Search                  1103            0.77   
Paid Search                     1394           

In [89]:
print(usage_events.columns.tolist())
print(usage_events.head(3).to_string())

['event_id', 'customer_id', 'event_timestamp', 'event_type', 'feature_category', 'session_duration']
                               event_id                           customer_id     event_timestamp             event_type feature_category  session_duration
0  19bc33ab-e93c-49f6-9b16-fcbe39d43da1  000cd21a-65a7-4a30-b5b7-0dd9728f5897 2026-01-12 15:52:41       Report Generated        Reporting               138
1  631ba69b-8a96-4211-927f-ea218039994a  000cd21a-65a7-4a30-b5b7-0dd9728f5897 2026-01-12 20:06:26  Custom Metric Created        Analytics               323
2  05ab9d77-0dc0-4b8b-8d9d-cb1c8016da2b  000cd21a-65a7-4a30-b5b7-0dd9728f5897 2026-01-23 19:40:22       Dashboard Viewed        Dashboard               214


In [90]:
usage_check = usage_events.copy()
usage_check["event_timestamp"] = pd.to_datetime(usage_check["event_timestamp"])

# Aggregate usage at the customer level
customer_usage = (
    usage_check.groupby("customer_id")
    .agg(
        total_events=("event_id", "count"),
        active_days=("event_timestamp", lambda x: x.dt.date.nunique()),
        avg_session_duration=("session_duration", "mean")
    )
    .reset_index()
)

# Calculate each customer's subscription exposure
exposure = subscriptions[
    ["customer_id", "start_date", "end_date", "subscription_status"]
].copy()

exposure["start_date"] = pd.to_datetime(exposure["start_date"])
exposure["end_date"] = pd.to_datetime(exposure["end_date"])

observation_end = pd.Timestamp("2026-08-01")

exposure["observation_date"] = exposure["end_date"].fillna(observation_end)
exposure["exposure_days"] = (
    exposure["observation_date"] - exposure["start_date"]
).dt.days.clip(lower=1)

exposure["exposure_months"] = exposure["exposure_days"] / 30.4375

# Combine and calculate normalized engagement
engagement = exposure.merge(customer_usage, on="customer_id", how="left")

engagement[["total_events", "active_days"]] = (
    engagement[["total_events", "active_days"]].fillna(0)
)

engagement["events_per_30_days"] = (
    engagement["total_events"] / engagement["exposure_months"]
)

engagement["active_days_per_30_days"] = (
    engagement["active_days"] / engagement["exposure_months"]
)

# Compare churned vs. active customers
engagement.groupby("subscription_status").agg(
    customers=("customer_id", "count"),
    avg_events_per_30_days=("events_per_30_days", "mean"),
    median_events_per_30_days=("events_per_30_days", "median"),
    avg_active_days_per_30_days=("active_days_per_30_days", "mean"),
    avg_session_duration=("avg_session_duration", "mean")
).round(2)

,customers,avg_events_per_30_days,median_events_per_30_days,avg_active_days_per_30_days,avg_session_duration
subscription_status,,,,,
Active,3654,9.84,9.41,4.93,452.53
Cancelled,1346,10.08,9.67,5.03,450.56


In [91]:
clean_dir = Path("churn_analysis_data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

customers.to_csv(clean_dir / "customers.csv", index=False)
subscriptions.to_csv(clean_dir / "subscriptions.csv", index=False)
subscription_events.to_csv(clean_dir / "subscription_events.csv", index=False)
invoices.to_csv(clean_dir / "invoices.csv", index=False)
support_tickets.to_csv(clean_dir / "support_tickets.csv", index=False)
usage_events.to_csv(clean_dir / "usage_events.csv", index=False)

list(clean_dir.iterdir())

[PosixPath('churn_analysis_data/clean/invoices.csv'),
 PosixPath('churn_analysis_data/clean/customers.csv'),
 PosixPath('churn_analysis_data/clean/usage_events.csv'),
 PosixPath('churn_analysis_data/clean/subscription_events.csv'),
 PosixPath('churn_analysis_data/clean/subscriptions.csv'),
 PosixPath('churn_analysis_data/clean/support_tickets.csv')]

In [92]:
dirty_dir = Path("churn_analysis_data/dirty")
dirty_dir.mkdir(parents=True, exist_ok=True)

customers_dirty = pd.read_csv("churn_analysis_data/clean/customers.csv")
subscriptions_dirty = pd.read_csv("churn_analysis_data/clean/subscriptions.csv")
subscription_events_dirty = pd.read_csv("churn_analysis_data/clean/subscription_events.csv")
invoices_dirty = pd.read_csv("churn_analysis_data/clean/invoices.csv")
support_tickets_dirty = pd.read_csv("churn_analysis_data/clean/support_tickets.csv")
usage_events_dirty = pd.read_csv("churn_analysis_data/clean/usage_events.csv")

dirty_tables = {
    "customers": customers_dirty, 
    "subscriptions": subscriptions_dirty, 
    "subscription_events": subscription_events_dirty, 
    "invoices": invoices_dirty, 
    "support_tickets": support_tickets_dirty,
    "usage_events": usage_events_dirty
}
for name, df in dirty_tables.items():
    print(f"{name}: {df.shape}")

customers: (5000, 6)
subscriptions: (5000, 10)
subscription_events: (1382, 8)
invoices: (36442, 11)
support_tickets: (4896, 10)
usage_events: (467065, 6)


In [93]:
customers_dirty.info()

customers_dirty.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          5000 non-null   object
 1   signup_date          5000 non-null   object
 2   acquisition_channel  5000 non-null   object
 3   country              5000 non-null   object
 4   industry             5000 non-null   object
 5   company_size         5000 non-null   object
dtypes: object(6)
memory usage: 234.5+ KB


customer_id            0
signup_date            0
acquisition_channel    0
country                0
industry               0
company_size           0
dtype: int64

In [94]:
customers_dirty.loc[
    customers_dirty.sample(frac=0.02, random_state=42).index,
    "industry"
] = np.nan

customers_dirty.loc[
    customers_dirty.sample(frac=0.02, random_state=43).index,
    "company_size"
] = np.nan

In [95]:
customers_dirty.isna().sum()

customer_id              0
signup_date              0
acquisition_channel      0
country                  0
industry               100
company_size           100
dtype: int64

In [96]:
customers_dirty.loc[
    customers_dirty.sample(frac=0.02, random_state=44).index,
    "country"
] = customers_dirty["country"].str.lower()

customers_dirty.loc[
    customers_dirty.sample(frac=0.02, random_state=45).index,
    "acquisition_channel"
] = customers_dirty["acquisition_channel"].str.strip().str.lower()

In [97]:
print(customers_dirty["country"].unique())
print(customers_dirty["acquisition_channel"].unique())

['Australia' 'US' 'UK' 'Canada' 'Germany' 'France' 'uk' 'us' 'france'
 'germany' 'australia' 'canada']
['Paid Search' 'Outbound Sales' 'Partner' 'Organic Search' 'Social Media'
 'Referral' 'partner' 'outbound sales' 'organic search' 'paid search'
 'referral' 'social media']


In [98]:
duplicate_customers = customers_dirty.sample(
    n=25,
    random_state=46
)

customers_dirty = pd.concat(
    [customers_dirty, duplicate_customers],
    ignore_index=True
)

In [99]:
print("Shape:", customers_dirty.shape)
print("Duplicate customer IDs:", customers_dirty["customer_id"].duplicated().sum())

Shape: (5025, 6)
Duplicate customer IDs: 25


In [100]:
print(subscriptions_dirty.shape)
print(subscriptions_dirty.isna().sum())

print(subscriptions_dirty["plan"].value_counts(dropna=False))
print(subscriptions_dirty["billing_cycle"].value_counts(dropna=False))
print(subscriptions_dirty["cancellation_reason"].value_counts(dropna=False))

(5000, 10)
subscription_id           0
customer_id               0
start_date                0
end_date               3654
subscription_status       0
billing_cycle             0
plan                      0
monthly_price             0
churn_type             3654
cancellation_reason    3654
dtype: int64
plan
Pro           2278
Basic         1920
Enterprise     802
Name: count, dtype: int64
billing_cycle
Monthly    3561
Annual     1439
Name: count, dtype: int64
cancellation_reason
NaN                             3654
Acquisition / Poor Fit           694
Repeated Payment Failure         299
Other / Changed Requirements     256
Poor Support Experience           97
Name: count, dtype: int64


In [101]:
plan_idx = subscriptions_dirty.sample(
    frac=0.02, random_state=47
).index

subscriptions_dirty.loc[plan_idx, "plan"] = (
    subscriptions_dirty.loc[plan_idx, "plan"].str.lower()
)

billing_idx = subscriptions_dirty.sample(
    frac=0.02, random_state=48
).index

subscriptions_dirty.loc[billing_idx, "billing_cycle"] = (
    subscriptions_dirty.loc[billing_idx, "billing_cycle"].str.lower()
)

In [102]:
print(subscriptions_dirty["plan"].value_counts())
print(subscriptions_dirty["billing_cycle"].value_counts())

plan
Pro           2236
Basic         1878
Enterprise     786
basic           42
pro             42
enterprise      16
Name: count, dtype: int64
billing_cycle
Monthly    3481
Annual     1419
monthly      80
annual       20
Name: count, dtype: int64


In [103]:
cancelled_idx = subscriptions_dirty[
    subscriptions_dirty["subscription_status"] == "Cancelled"
].sample(frac=0.02, random_state=49).index

subscriptions_dirty.loc[
    cancelled_idx, "cancellation_reason"
] = np.nan

In [104]:
print(
    subscriptions_dirty.loc[
        subscriptions_dirty["subscription_status"] == "Cancelled",
        "cancellation_reason"
    ].isna().sum()
)

27


In [105]:
duplicate_subscriptions = subscriptions_dirty.sample(
    n=25,
    random_state=50
)

subscriptions_dirty = pd.concat(
    [subscriptions_dirty, duplicate_subscriptions],
    ignore_index=True
)

In [106]:
print("Shape:", subscriptions_dirty.shape)
print(
    "Duplicate subscription IDs:",
    subscriptions_dirty["subscription_id"].duplicated().sum()
)

Shape: (5025, 10)
Duplicate subscription IDs: 25


In [107]:
idx = subscription_events_dirty.sample(
    frac=0.03, random_state=51
).index

subscription_events_dirty.loc[idx, "event_type"] = (
    subscription_events_dirty.loc[idx, "event_type"].str.lower()
)


idx = subscription_events_dirty.sample(
    frac=0.02, random_state=52
).index

subscription_events_dirty.loc[idx, "previous_plan"] = np.nan


idx = subscription_events_dirty.sample(
    frac=0.02, random_state=53
).index

subscription_events_dirty.loc[idx, "event_date"] = (
    pd.to_datetime(subscription_events_dirty.loc[idx, "event_date"])
    .dt.strftime("%m/%d/%Y")
)


duplicates = subscription_events_dirty.sample(
    frac=0.005, random_state=54
)

subscription_events_dirty = pd.concat(
    [subscription_events_dirty, duplicates],
    ignore_index=True
)

In [108]:
print("Shape:", subscription_events_dirty.shape)

print(
    "Duplicate event IDs:",
    subscription_events_dirty["event_id"].duplicated().sum()
)

print("\nMissing values:")
print(subscription_events_dirty.isna().sum())

print("\nEvent types:")
print(subscription_events_dirty["event_type"].value_counts())

print("\nSample dates:")
print(subscription_events_dirty["event_date"].sample(10, random_state=42))

Shape: (1389, 8)
Duplicate event IDs: 7

Missing values:
event_id            0
subscription_id     0
event_date          0
event_type          0
previous_plan      28
new_plan            0
previous_price      0
new_price           0
dtype: int64

Event types:
event_type
Upgrade      955
Downgrade    393
upgrade       25
downgrade     16
Name: count, dtype: int64

Sample dates:
558     2026-07-16
168     2026-07-06
240     2026-02-15
664     2026-03-06
271     2025-03-21
585     2025-07-14
1328    2025-08-13
552     2025-10-12
901     2026-03-29
1048    2026-03-20
Name: event_date, dtype: object


In [109]:
idx = invoices_dirty.sample(
    frac=0.02, random_state=55
).index

invoices_dirty.loc[idx, "payment_status"] = (
    invoices_dirty.loc[idx, "payment_status"].str.lower()
)


eligible = invoices_dirty[
    invoices_dirty["failure_reason"].notna()
]

idx = eligible.sample(
    frac=0.02, random_state=56
).index

invoices_dirty.loc[idx, "failure_reason"] = np.nan


idx = invoices_dirty.sample(
    frac=0.003, random_state=57
).index

invoices_dirty.loc[idx, "amount"] = (
    -invoices_dirty.loc[idx, "amount"].abs()
)


idx = invoices_dirty.sample(
    frac=0.01, random_state=58
).index

invoices_dirty.loc[idx, "retry_count"] = np.nan


duplicates = invoices_dirty.sample(
    frac=0.005, random_state=59
)

invoices_dirty = pd.concat(
    [invoices_dirty, duplicates],
    ignore_index=True
)

In [110]:
print("Shape:", invoices_dirty.shape)

print(
    "Duplicate invoice IDs:",
    invoices_dirty["invoice_id"].duplicated().sum()
)

print("\nMissing values:")
print(invoices_dirty.isna().sum())

print("\nPayment statuses:")
print(invoices_dirty["payment_status"].value_counts())

print(
    "\nNegative amounts:",
    (invoices_dirty["amount"] < 0).sum()
)

print(
    "\nMissing failure reasons on invoices with retries:",
    invoices_dirty.loc[
        invoices_dirty["retry_count"].gt(0),
        "failure_reason"
    ].isna().sum()
)

Shape: (36624, 11)
Duplicate invoice IDs: 182

Missing values:
invoice_id               0
subscription_id          0
invoice_date             0
due_date                 0
amount                   0
payment_status           0
payment_date          2799
retry_count            367
failure_reason       35048
recovered                0
last_attempt_date     2496
dtype: int64

Payment statuses:
payment_status
Paid       33150
Pending     2441
paid         675
Failed       300
pending       55
failed         3
Name: count, dtype: int64

Negative amounts: 110

Missing failure reasons on invoices with retries: 32


In [111]:
idx = support_tickets_dirty.sample(
    frac=0.03, random_state=60
).index

support_tickets_dirty.loc[idx, "issue_category"] = (
    support_tickets_dirty.loc[idx, "issue_category"].str.lower()
)


idx = support_tickets_dirty.sample(
    frac=0.02, random_state=61
).index

support_tickets_dirty.loc[idx, "priority"] = (
    support_tickets_dirty.loc[idx, "priority"].str.lower()
)


resolved = support_tickets_dirty[
    support_tickets_dirty["ticket_status"] == "Resolved"
]

idx = resolved.sample(
    frac=0.02, random_state=62
).index

support_tickets_dirty.loc[idx, "resolution_time_hours"] = np.nan


resolved = support_tickets_dirty[
    support_tickets_dirty["ticket_status"] == "Resolved"
]

idx = resolved.sample(
    frac=0.005, random_state=63
).index

support_tickets_dirty.loc[idx, "resolution_time_hours"] = (
    -support_tickets_dirty.loc[idx, "resolution_time_hours"].abs()
)


duplicates = support_tickets_dirty.sample(
    frac=0.005, random_state=64
)

support_tickets_dirty = pd.concat(
    [support_tickets_dirty, duplicates],
    ignore_index=True
)

In [112]:
print("Shape:", support_tickets_dirty.shape)

print(
    "Duplicate ticket IDs:",
    support_tickets_dirty["ticket_id"].duplicated().sum()
)

print("\nMissing values:")
print(support_tickets_dirty.isna().sum())

print("\nIssue categories:")
print(support_tickets_dirty["issue_category"].value_counts())

print("\nPriorities:")
print(support_tickets_dirty["priority"].value_counts())

print(
    "\nNegative resolution times:",
    (support_tickets_dirty["resolution_time_hours"] < 0).sum()
)

print(
    "\nResolved tickets with missing resolution times:",
    support_tickets_dirty.loc[
        support_tickets_dirty["ticket_status"] == "Resolved",
        "resolution_time_hours"
    ].isna().sum()
)

Shape: (4920, 10)
Duplicate ticket IDs: 24

Missing values:
ticket_id                  0
customer_id                0
created_at                 0
resolved_at               36
issue_category             0
priority                   0
ticket_status              0
resolution_time_hours    134
sla_hours                  0
sla_breached               0
dtype: int64

Issue categories:
issue_category
Technical Issue       1197
Integration            899
Product Question       861
Billing                711
Account Management     552
Data Issue             552
technical issue         38
integration             36
account management      22
product question        22
billing                 17
data issue              13
Name: count, dtype: int64

Priorities:
priority
Medium      2420
Low         1185
High         962
Critical     255
medium        41
low           29
high          21
critical       7
Name: count, dtype: int64

Negative resolution times: 23

Resolved tickets with missing resolut

In [113]:
idx = usage_events_dirty.sample(
    frac=0.02, random_state=65
).index

usage_events_dirty.loc[idx, "event_type"] = (
    usage_events_dirty.loc[idx, "event_type"].str.lower()
)


idx = usage_events_dirty.sample(
    frac=0.02, random_state=66
).index

usage_events_dirty.loc[idx, "feature_category"] = np.nan


idx = usage_events_dirty.sample(
    frac=0.002, random_state=67
).index

usage_events_dirty.loc[idx, "session_duration"] = (
    -usage_events_dirty.loc[idx, "session_duration"].abs()
)


idx = usage_events_dirty.sample(
    frac=0.01, random_state=68
).index

usage_events_dirty.loc[idx, "session_duration"] = np.nan


duplicates = usage_events_dirty.sample(
    frac=0.005, random_state=69
)

usage_events_dirty = pd.concat(
    [usage_events_dirty, duplicates],
    ignore_index=True
)

In [114]:
print("Shape:", usage_events_dirty.shape)

print(
    "Duplicate event IDs:",
    usage_events_dirty["event_id"].duplicated().sum()
)

print("\nMissing values:")
print(usage_events_dirty.isna().sum())

print("\nEvent types:")
print(usage_events_dirty["event_type"].value_counts())

print(
    "\nNegative session durations:",
    (usage_events_dirty["session_duration"] < 0).sum()
)

print(
    "\nMissing session durations:",
    usage_events_dirty["session_duration"].isna().sum()
)

Shape: (469400, 6)
Duplicate event IDs: 2335

Missing values:
event_id               0
customer_id            0
event_timestamp        0
event_type             0
feature_category    9390
session_duration    4694
dtype: int64

Event types:
event_type
Dashboard Viewed         174955
Report Generated          83132
Integration Used          55426
Data Exported             36541
Custom Metric Created     36514
Alert Configured          32097
Team Member Invited       22926
Integration Connected     18425
dashboard viewed           3564
report generated           1699
integration used           1101
data exported               757
custom metric created       737
alert configured            645
team member invited         496
integration connected       385
Name: count, dtype: int64

Negative session durations: 924

Missing session durations: 4694


In [115]:
dirty_tables = {
    "customers": customers_dirty,
    "subscriptions": subscriptions_dirty,
    "subscription_events": subscription_events_dirty,
    "invoices": invoices_dirty,
    "support_tickets": support_tickets_dirty,
    "usage_events": usage_events_dirty
}

for name, df in dirty_tables.items():
    df.to_csv(dirty_dir / f"{name}.csv", index=False)
    print(f"Saved {name}: {df.shape}")

Saved customers: (5025, 6)
Saved subscriptions: (5025, 10)
Saved subscription_events: (1389, 8)
Saved invoices: (36624, 11)
Saved support_tickets: (4920, 10)
Saved usage_events: (469400, 6)
